In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
features = pd.read_parquet(
    "../data/processed/features_basic.parquet"
)

features["date"] = pd.to_datetime(features["date"])

features.head()

,territory_id,date,health,marketplaces,catering,food,transport,total_spending,food_share,health_share,marketplaces_share,catering_share,transport_share,market_access
0,1,2023-01-01,1271,2505,1142,7692,1718,14328,0.536851,0.088707,0.174832,0.079704,0.119905,309.6
1,1,2023-02-01,1276,2748,1095,8259,1704,15082,0.547606,0.084604,0.182204,0.072603,0.112982,309.6
2,1,2023-03-01,1410,3052,1256,8873,1847,16438,0.539786,0.085777,0.185667,0.076408,0.112362,309.6
3,1,2023-04-01,1277,2850,1171,8648,1784,15730,0.549777,0.081182,0.181182,0.074444,0.113414,309.6
4,1,2023-05-01,1295,2994,1315,8738,1817,16159,0.540751,0.080141,0.185284,0.081379,0.112445,309.6


In [3]:
model_features = [
    "food_share",
    "health_share",
    "marketplaces_share",
    "catering_share",
    "transport_share"
]

In [4]:
target_date = pd.Timestamp("2024-12-01")

month_data = (
    features[features["date"] == target_date]
    .copy()
)

month_data.shape

(2103, 14)

In [5]:
X = month_data[model_features].to_numpy()

municipalities = month_data["territory_id"].to_numpy()

print("Размер матрицы:", X.shape)
print("Количество муниципалитетов:", len(municipalities))

Размер матрицы: (2103, 5)
Количество муниципалитетов: 2103


In [6]:
similarity_matrix = cosine_similarity(X)

similarity_matrix.shape

(2103, 2103)

In [7]:
similarity_matrix[:5, :5]

array([[1.        , 0.99769454, 0.99077344, 0.99778811, 0.9937315 ],
       [0.99769454, 1.        , 0.99369541, 0.99444383, 0.9914984 ],
       [0.99077344, 0.99369541, 1.        , 0.99396718, 0.9968313 ],
       [0.99778811, 0.99444383, 0.99396718, 1.        , 0.99858141],
       [0.9937315 , 0.9914984 , 0.9968313 , 0.99858141, 1.        ]])

In [8]:
upper_triangle = similarity_matrix[
    np.triu_indices_from(similarity_matrix, k=1)
]

print(pd.Series(upper_triangle).describe())

count    2.210253e+06
mean     9.850643e-01
std      1.981695e-02
min      5.869938e-01
25%      9.809970e-01
50%      9.915456e-01
75%      9.968107e-01
max      9.999986e-01
dtype: float64


In [9]:
from sklearn.metrics import pairwise_distances

In [10]:
distance_matrix = pairwise_distances(
    X,
    metric="euclidean"
)

distance_matrix.shape

(2103, 2103)

In [11]:
upper_triangle_distance = distance_matrix[
    np.triu_indices_from(distance_matrix, k=1)
]

print(pd.Series(upper_triangle_distance).describe())

count    2.210253e+06
mean     1.079336e-01
std      6.604350e-02
min      1.189539e-03
25%      5.836392e-02
50%      9.481480e-02
75%      1.441970e-01
max      7.040338e-01
dtype: float64


In [12]:
correlation_matrix = np.corrcoef(X)

correlation_matrix.shape

(2103, 2103)

In [13]:
upper_triangle_corr = correlation_matrix[
    np.triu_indices_from(correlation_matrix, k=1)
]

print(pd.Series(upper_triangle_corr).describe())

count    2.210253e+06
mean     9.809092e-01
std      3.399107e-02
min      3.208719e-01
25%      9.801068e-01
50%      9.917192e-01
75%      9.967537e-01
max      9.999999e-01
dtype: float64


In [14]:
from sklearn.neighbors import NearestNeighbors

In [15]:
k = 10

knn = NearestNeighbors(
    n_neighbors=k + 1,
    metric="euclidean"
)

knn.fit(X)

distances, indices = knn.kneighbors(X)

print("Размер distances:", distances.shape)
print("Размер indices:", indices.shape)

Размер distances: (2103, 11)
Размер indices: (2103, 11)


In [16]:
neighbor_distances = distances[:, 1:]

print(
    pd.Series(neighbor_distances.ravel()).describe()
)

count    21030.000000
mean         0.013758
std          0.009945
min          0.001190
25%          0.008648
50%          0.011237
75%          0.014974
max          0.127214
dtype: float64


In [17]:
edge_weights = 1 / (1 + distances[:, 1:])

print(
    pd.Series(edge_weights.ravel()).describe()
)

count    21030.000000
mean         0.986520
std          0.009357
min          0.887143
25%          0.985246
50%          0.988887
75%          0.991426
max          0.998812
dtype: float64


In [18]:
edges = []

for i in range(len(municipalities)):
    for j in range(k):
        neighbor_idx = indices[i, j + 1]
        weight = edge_weights[i, j]

        edges.append({
            "source": int(municipalities[i]),
            "target": int(municipalities[neighbor_idx]),
            "weight": float(weight)
        })

edges = pd.DataFrame(edges)

edges.head()

,source,target,weight
0,1,2597,0.988610
1,1,819,0.984544
2,1,813,0.981706
3,1,1695,0.981532
4,1,1453,0.980007


In [19]:
sigma = np.median(neighbor_distances)

print("sigma =", sigma)

sigma = 0.011237473945055972


In [20]:
edge_weights_rbf = np.exp(
    -neighbor_distances / sigma
)

print(
    pd.Series(edge_weights_rbf.ravel()).describe()
)

count    21030.000000
mean         0.359573
std          0.158760
min          0.000012
25%          0.263805
50%          0.367879
75%          0.463221
max          0.899555
dtype: float64


In [21]:
edges_rbf = []

for i in range(len(municipalities)):
    for j in range(k):
        neighbor_idx = indices[i, j + 1]

        edges_rbf.append({
            "source": int(municipalities[i]),
            "target": int(municipalities[neighbor_idx]),
            "weight": float(edge_weights_rbf[i, j])
        })

edges_rbf = pd.DataFrame(edges_rbf)

edges_rbf.head()

,source,target,weight
0,1,2597,0.358711
1,1,819,0.247334
2,1,813,0.190467
3,1,1695,0.187424
4,1,1453,0.162773


In [22]:
edges_rbf["node_min"] = edges_rbf[["source", "target"]].min(axis=1)
edges_rbf["node_max"] = edges_rbf[["source", "target"]].max(axis=1)

edges_undirected = (
    edges_rbf
    .groupby(["node_min", "node_max"], as_index=False)["weight"]
    .mean()
    .rename(
        columns={
            "node_min": "source",
            "node_max": "target"
        }
    )
)

edges_undirected.head()

,source,target,weight
0,1,811,0.148837
1,1,812,0.153565
2,1,813,0.190467
3,1,819,0.247334
4,1,1117,0.112621


In [23]:
print("Муниципалитетов:", len(municipalities))
print("Направленных связей:", len(edges_rbf))
print("Ненаправленных связей:", len(edges_undirected))
print("Средняя степень:", 2 * len(edges_undirected) / len(municipalities))

Муниципалитетов: 2103
Направленных связей: 21030
Ненаправленных связей: 13757
Средняя степень: 13.083214455539705


In [24]:
import networkx as nx

In [25]:
G = nx.Graph()

G.add_nodes_from(municipalities)

for row in edges_undirected.itertuples(index=False):
    G.add_edge(
        int(row.source),
        int(row.target),
        weight=float(row.weight)
    )

print("Количество узлов:", G.number_of_nodes())
print("Количество рёбер:", G.number_of_edges())

Количество узлов: 2103
Количество рёбер: 13757


In [26]:
components = list(nx.connected_components(G))

print("Количество компонент:", len(components))

component_sizes = sorted(
    [len(component) for component in components],
    reverse=True
)

print("\nРазмеры крупнейших компонент:")
print(component_sizes[:20])

Количество компонент: 1

Размеры крупнейших компонент:
[2103]


In [27]:
largest_component_share = (
    component_sizes[0] / G.number_of_nodes()
)

print(
    "Доля крупнейшей компоненты:",
    largest_component_share
)

Доля крупнейшей компоненты: 1.0


In [28]:
degrees = [degree for _, degree in G.degree()]

print(pd.Series(degrees).describe())

count    2103.000000
mean       13.083214
std         2.298019
min        10.000000
25%        11.000000
50%        13.000000
75%        15.000000
max        22.000000
dtype: float64


In [29]:
edges_undirected.to_parquet(
    "../data/processed/edges_2024_12.parquet",
    index=False
)

month_data[
    ["territory_id"] + model_features
].to_parquet(
    "../data/processed/features_2024_12.parquet",
    index=False
)

print("Сеть и признаки сохранены.")

Сеть и признаки сохранены.
